# Phase 5: Temporal Feature Engineering Pipeline

**Project:** Smart Student Retention & Early Warning System  
**Prediction Cutoff:** Relative Day 35 ($t \le 35$ after presentation start)  
**Primary Cohort:** Policy C (Active at start, survived to Day 35; $N = 27,128$)  
**Target Variable:** `withdrawn_after_cutoff` (Withdrawal occurring strictly after relative Day 35; Base rate: 17.39%)  

---

## 1. Architectural Principles & Strict Anti-Leakage Constraints

1. **Strict Temporal Horizon Invariant ($t \le 35$):** All predictive features must derive exclusively from interactions and milestones observable on or before relative Day 35. Activity after Day 35 is strictly excluded to prevent retrospective lookahead leakage.
2. **Strict Outcome Variable Exclusion:** Neither `final_result`, `date_unregistration`, nor any derived indicator of eventual withdrawal may appear as model features.
3. **Tutor Turnaround & Score Exclusion:** `studentAssessment.score` records marks awarded by tutors or computers. However, OULAD does not record the return date when grades were released to students or advisors. Because score availability at Day 35 cannot be proven, assessment scores are excluded from the primary Day-35 baseline to prevent latency leakage.
4. **Locked 22-Feature Allowlist:** Predictive modeling is strictly restricted to 22 non-demographic features across 4 functional domains:
   - **Course Context (4):** `code_module`, `presentation_season`, `studied_credits`, `num_of_prev_attempts`
   - **Registration Timing (2):** `registration_lead_time_days`, `is_late_registration`
   - **VLE Digital Engagement (11):** `vle_total_clicks_window`, `vle_active_days_window`, `vle_unique_sites_accessed`, `vle_pre_course_clicks`, `vle_recent_clicks_14d`, `vle_velocity_ratio`, `days_since_last_vle_interaction`, `has_no_vle_activity_by_cutoff`, `vle_clicks_forumng`, `vle_clicks_oucontent`, `vle_clicks_quiz`
   - **Assessment Progress (5):** `assessments_due_count`, `submissions_count_lte_cutoff`, `submitted_on_time_count`, `late_submissions_count_lte_cutoff`, `missing_due_submissions_count`
5. **Unobserved Inactivity Representation:** For learners with 0 recorded interactions by Day 35, `days_since_last_vle_interaction` is assigned the upper-bound sentinel value **65** (strictly greater than maximum possible gap of 60 days) and companion indicator `has_no_vle_activity_by_cutoff = 1`.
6. **Due Coursework Universe ($A_{\text{due}}$):** Restricted to continuous non-exam coursework with scheduled deadlines $\le 35$. Early submissions for future assignments ($> 35$) are excluded from due counts to prevent anomalies where on-time submissions exceed due assessments.


In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    """Locate project root by searching upwards for pyproject.toml."""
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / "pyproject.toml").is_file():
            return directory
    return current.parent if (current.parent / "pyproject.toml").is_file() else current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
COHORT_PATH = DATA_DIR / "processed" / "cohort" / "cohort_policy_c.parquet"
INTERIM_DIR = DATA_DIR / "interim" / "normalized"
FEATURES_DIR = DATA_DIR / "processed" / "features"

print(f"Project root: {PROJECT_ROOT}")
print(f"Cohort path:  {COHORT_PATH} (exists: {COHORT_PATH.exists()})")
print(f"Interim dir:  {INTERIM_DIR} (exists: {INTERIM_DIR.exists()})")
print(f"Features dir: {FEATURES_DIR} (exists: {FEATURES_DIR.exists()})")


## 2. Inspecting the Primary Policy C Cohort

The Policy C cohort consists of 27,128 active learners who survived through the prediction cutoff (Day 35).


In [ ]:
cohort_df = pd.read_parquet(COHORT_PATH)
print(f"Total eligible observations: {len(cohort_df):,}")
print("Target class distribution ('withdrawn_after_cutoff'):")
counts = cohort_df['withdrawn_after_cutoff'].value_counts().sort_index()
for val, count in counts.items():
    label = 'Retained (Pass/Fail/Distinction)' if val == 0 else 'Withdrawn after Day 35'
    pct = count / len(cohort_df) * 100
    print(f"  Class {val} ({label}): {count:,} ({pct:.2f}%)")


## 3. Feature Group 1: Course Context Features (4 Predictors)

Contextual entry attributes:
- `code_module`: Nominal course identifier (`AAA`–`GGG`)
- `presentation_season`: Stable seasonal indicator (`B` for Feb/Spring, `J` for Oct/Autumn) replacing year-specific strings to prevent out-of-time one-hot zero-vector collapse
- `studied_credits`: Total concurrent credits studied (Median: 60)
- `num_of_prev_attempts`: Number of prior attempts on this module (0 to 6)


In [ ]:
context_df = cohort_df[['code_module', 'code_presentation', 'id_student', 'studied_credits', 'num_of_prev_attempts']].copy()
context_df['presentation_season'] = context_df['code_presentation'].str[-1]

print("Seasonal distribution across cohort:")
print(context_df['presentation_season'].value_counts())

print("\nSummary statistics of quantitative context features:")
print(context_df[['studied_credits', 'num_of_prev_attempts']].describe())


## 4. Feature Group 2: Registration Timing Features (2 Predictors)

- `registration_lead_time_days`: $-\text{date\_registration}$. Positive for advance registration (e.g. Day -30 $\to$ +30 days), zero for Day 0, negative for late registration.
- `is_late_registration`: Binary flag ($1$ if $0 < \text{date\_registration} \le 35$, else $0$). Evaluates to NaN if registration date is unrecorded under Policy B.


In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from student_retention.transformation.registration_features import (
    extract_registration_features,
)

reg_feats = extract_registration_features(cohort_df=cohort_df, cutoff_day=35)
print("Registration feature distributions:")
print(reg_feats[['registration_lead_time_days', 'is_late_registration']].describe())
print(f"\nLate registration count (registered Day 1 to 35): {int(reg_feats['is_late_registration'].sum()):,} students")


## 5. Feature Group 3: VLE Digital Engagement Features (11 Predictors)

All VLE features are computed using only interaction logs with $\text{date} \le 35$:
- Cumulative intensity: `vle_total_clicks_window`, `vle_active_days_window`, `vle_unique_sites_accessed`
- Temporal window dynamics: `vle_pre_course_clicks` ($[-25, -1]$), `vle_recent_clicks_14d` ($[22, 35]$)
- Acceleration ratio: $\text{vle\_velocity\_ratio} = \frac{\text{clicks}_{[29, 35]} / 7}{(\text{clicks}_{[0, 28]} / 29) + 1.0}$
- Inactivity representation: `days_since_last_vle_interaction` ($35 - \max(\text{date})$, sentinel 65 for 0 clicks) and `has_no_vle_activity_by_cutoff`
- Resource types: `vle_clicks_forumng`, `vle_clicks_oucontent`, `vle_clicks_quiz`


In [ ]:
student_vle_df = pd.read_parquet(INTERIM_DIR / "student_vle.parquet")
vle_df = pd.read_parquet(INTERIM_DIR / "vle.parquet")
from student_retention.transformation.vle_features import extract_vle_features

vle_feats = extract_vle_features(cohort_df=cohort_df, student_vle_df=student_vle_df, vle_df=vle_df, cutoff_day=35)
vle_cols = [c for c in vle_feats.columns if c.startswith(('vle_', 'days_', 'has_'))]
print(f"Extracted {len(vle_cols)} VLE engagement features. Summary statistics:")
print(vle_feats[vle_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])


## 6. Feature Group 4: Assessment Progress Features (5 Predictors)

Bounded to continuous non-exam coursework due on or before Day 35 ($A_{\text{due}}$):
- `assessments_due_count`: Scheduled due non-exam coursework (adjusted for late registration opportunity)
- `submissions_count_lte_cutoff`: Genuine submissions by cutoff ($\text{date\_submitted} \le 35 \land \text{is\_banked} == 0$)
- `submitted_on_time_count`: Genuine submissions turned in on or before deadline
- `late_submissions_count_lte_cutoff`: Genuine submissions turned in after deadline but $\le 35$
- `missing_due_submissions_count`: Due assessments lacking both genuine submission and banked credit

**Partition Invariant:** $\text{submitted\_on\_time\_count} + \text{late\_submissions\_count\_lte\_cutoff} = \text{submissions\_count\_lte\_cutoff} \le \text{assessments\_due\_count}$


In [ ]:
student_assessment_df = pd.read_parquet(INTERIM_DIR / "student_assessment.parquet")
assessments_df = pd.read_parquet(INTERIM_DIR / "assessments.parquet")
from student_retention.transformation.assessment_features import (
    extract_assessment_features,
)

assess_feats = extract_assessment_features(cohort_df=cohort_df, student_assessment_df=student_assessment_df, assessments_df=assessments_df, cutoff_day=35)
assess_cols = ['assessments_due_count', 'submissions_count_lte_cutoff', 'submitted_on_time_count', 'late_submissions_count_lte_cutoff', 'missing_due_submissions_count']
print(f"Extracted {len(assess_cols)} assessment progress features. Summary statistics:")
print(assess_feats[assess_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])

# Verify partition invariant
part_diff = assess_feats['submissions_count_lte_cutoff'] - (assess_feats['submitted_on_time_count'] + assess_feats['late_submissions_count_lte_cutoff'])
print(f"\nPartition invariant violations: {int((part_diff != 0).sum())} (Expected: 0)")


## 7. Full Feature Matrix Assembly & Quality Validation Engine

We inspect the persisted production feature matrix `feature_matrix_policy_c.parquet` and verify quality rules.


In [ ]:
from student_retention.features.feature_definitions import PRIMARY_FEATURE_ALLOWLIST
from student_retention.features.feature_validation import validate_feature_table

matrix_path = FEATURES_DIR / "feature_matrix_policy_c.parquet"
feature_df = pd.read_parquet(matrix_path)
print(f"Loaded production feature matrix: {len(feature_df):,} rows x {len(feature_df.columns)} columns")

# Run programmatic validation engine
report = validate_feature_table(feature_df, expected_row_count=27128, policy_version="Policy C")
print(f"Validation Status: {report['metrics']['validation_status']}")
print(f"Duplicate grain keys: {report['metrics']['duplicate_grain_count']}")
print(f"Missing allowlist features: {len(report['metrics']['missing_allowlist_features'])}")
print(f"Total feature nulls: {sum(report['metrics']['null_counts'].values())}")
print(f"Target base rate: {report['metrics']['target_positive_base_rate']:.4%}")


## 8. Empirical Anti-Leakage Verification

Demonstrate that events occurring strictly after Day 35 cannot alter Day 35 features.


In [ ]:
from student_retention.features.feature_selection import verify_no_leakage_columns

# 1. Verify absence of forbidden columns from feature allowlist
verify_no_leakage_columns(PRIMARY_FEATURE_ALLOWLIST)
print("Passed: Zero forbidden leakage columns in PRIMARY_FEATURE_ALLOWLIST.")

# 2. Demonstrate post-cutoff VLE event invariance
sample_student = cohort_df.iloc[[0]]
vle_base = student_vle_df[student_vle_df['date'] <= 35].head(5).copy()
vle_leak = vle_base.copy()
# Inject post-cutoff event
injected_event = pd.DataFrame({
    'code_module': [sample_student['code_module'].iloc[0]],
    'code_presentation': [sample_student['code_presentation'].iloc[0]],
    'id_student': [sample_student['id_student'].iloc[0]],
    'id_site': [1],
    'date': [40],  # Day 40 > Day 35
    'sum_click': [1000],
})
vle_leak = pd.concat([vle_leak, injected_event], ignore_index=True)

f_base = extract_vle_features(sample_student, vle_base, vle_df, cutoff_day=35)
f_leak = extract_vle_features(sample_student, vle_leak, vle_df, cutoff_day=35)

is_identical = f_base[vle_cols].equals(f_leak[vle_cols])
print(f"Post-cutoff event invariance: {is_identical} (Day 40 activity had 0 impact on Day 35 features)")


---

## 9. Summary & Handoff to Phase 6 Modeling

- **Standardized Matrix:** `feature_matrix_policy_c.parquet` ($27,128 \times 31$) successfully created and validated.
- **Invariants Verified:** Zero unexpected nulls, exact 1-to-1 canonical observation grain, non-negative bounds, counter partition equality ($on\_time + late = submissions$), and inactivity sentinel value 65.
- **Temporal Security:** Anti-leakage tests verify strict cutoff enforcement.
- **Next Phase:** Phase 6 — Baseline, Modeling and Evaluation (DummyClassifier baseline, Logistic Regression, Random Forest, group-aware cross-validation blocking on student identifiers).
